**Synthetic data generation.** Segment-level treatment effects are hard-coded in the generator (At Risk +0.18, Lapsed +0.10, New +0.08, Active +0.05 before noise), so the experiment tests whether the analysis pipeline can recover a known structure. All data are synthetic.

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Make results reproducible
np.random.seed(42)
random.seed(42)

print("Setup complete!")

Setup complete!


In [2]:
# Cell 2 — Generate customers

N_CUSTOMERS = 15_000

customer_ids = np.arange(1, N_CUSTOMERS + 1)

# Historical period: 180 days
end_date = pd.Timestamp("2026-09-30")
start_date = end_date - pd.Timedelta(days=180)

customers = pd.DataFrame({
    "customer_id": customer_ids,

    "signup_date": pd.to_datetime(
        np.random.randint(
            (end_date - pd.Timedelta(days=365)).value // 10**9,
            end_date.value // 10**9,
            N_CUSTOMERS
        ),
        unit="s"
    ),

    "acquisition_channel": np.random.choice(
        ["Organic", "Paid Search", "Social", "Referral", "Partnership"],
        size=N_CUSTOMERS,
        p=[0.30, 0.25, 0.20, 0.15, 0.10]
    ),

    "geography": np.random.choice(
        ["Northeast", "South", "Midwest", "West"],
        size=N_CUSTOMERS,
        p=[0.23, 0.30, 0.22, 0.25]
    )
})

# Calculate customer tenure
customers["customer_tenure_days"] = (
    end_date - customers["signup_date"]
).dt.days

customers.head()

,customer_id,signup_date,acquisition_channel,geography,customer_tenure_days
0,1,2026-06-25 12:36:44,Organic,South,96
1,2,2026-04-04 06:41:18,Social,South,178
2,3,2026-06-26 23:57:30,Social,Northeast,95
3,4,2026-08-06 20:42:47,Organic,South,54
4,5,2026-06-01 00:03:08,Partnership,South,120


In [3]:
print("Number of customers:", len(customers))
print("\nAcquisition channels:")
print(customers["acquisition_channel"].value_counts())

print("\nGeography:")
print(customers["geography"].value_counts())

print("\nTenure summary:")
print(customers["customer_tenure_days"].describe())

Number of customers: 15000

Acquisition channels:
acquisition_channel
Organic        4381
Paid Search    3792
Social         2996
Referral       2299
Partnership    1532
Name: count, dtype: int64

Geography:
geography
South        4531
West         3620
Northeast    3481
Midwest      3368
Name: count, dtype: int64

Tenure summary:
count    15000.000000
mean       182.046600
std        105.692203
min          0.000000
25%         91.000000
50%        181.000000
75%        273.000000
max        364.000000
Name: customer_tenure_days, dtype: float64


In [4]:
# Cell 4 — Generate realistic customer behavior and orders

orders_list = []

categories = ["Category A", "Category B", "Category C", "Category D", "Category E"]
channels = ["App", "Web"]

for _, customer in customers.iterrows():

    customer_id = customer["customer_id"]

    # Base activity level for this customer
    # Some customers naturally order more frequently than others
    activity_level = np.random.gamma(shape=2.0, scale=0.8)

    # Customers with longer tenure have had more opportunity to order
    tenure_factor = min(customer["customer_tenure_days"] / 180, 1.5)

    # Expected number of orders over the 180-day period
    expected_orders = activity_level * tenure_factor * 5

    # Generate actual order count
    n_orders = np.random.poisson(expected_orders)

    if n_orders == 0:
        continue

    # Generate order dates
    customer_start = max(
        customer["signup_date"],
        start_date
    )

    available_days = max(
        (end_date - customer_start).days,
        1
    )

    order_days = np.random.randint(
        0,
        available_days + 1,
        n_orders
    )

    for day in order_days:

        order_date = customer_start + pd.Timedelta(days=int(day))

        # Base order value
        order_value = np.random.lognormal(
            mean=np.log(28),
            sigma=0.45
        )

        # Occasional larger orders
        if np.random.random() < 0.12:
            order_value *= np.random.uniform(1.5, 2.5)

        order_value = round(order_value, 2)

        # Some customers/orders receive discounts
        discount_probability = 0.25

        if np.random.random() < discount_probability:
            discount_amount = round(
                np.random.choice([2, 3, 5, 8]),
                2
            )
        else:
            discount_amount = 0.0

        net_order_value = round(
            max(order_value - discount_amount, 0),
            2
        )

        orders_list.append({
            "customer_id": customer_id,
            "order_date": order_date,
            "order_value": order_value,
            "discount_amount": discount_amount,
            "net_order_value": net_order_value,
            "category": np.random.choice(categories),
            "channel": np.random.choice(
                channels,
                p=[0.75, 0.25]
            )
        })


orders = pd.DataFrame(orders_list)

# Create unique order IDs
orders.insert(
    0,
    "order_id",
    np.arange(1, len(orders) + 1)
)

# Sort
orders = orders.sort_values(
    ["customer_id", "order_date"]
).reset_index(drop=True)

print("Orders generated:", len(orders))
orders.head()

Orders generated: 112932


,order_id,customer_id,order_date,order_value,discount_amount,net_order_value,category,channel
0,1,2,2026-04-05 06:41:18,29.98,0.0,29.98,Category D,App
1,5,2,2026-04-10 06:41:18,22.88,0.0,22.88,Category C,Web
2,8,2,2026-04-12 06:41:18,59.02,0.0,59.02,Category C,Web
3,6,2,2026-05-11 06:41:18,53.22,0.0,53.22,Category A,App
4,3,2,2026-05-14 06:41:18,36.41,2.0,34.41,Category D,App


In [5]:
# Cell 5 — Inspect order data

print("Number of orders:", len(orders))
print("Unique customers with orders:", orders["customer_id"].nunique())

print("\nOrder value:")
print(orders["order_value"].describe())

print("\nDiscount rate:")
print((orders["discount_amount"] > 0).mean())

print("\nOrders by channel:")
print(orders["channel"].value_counts())

print("\nOrders by category:")
print(orders["category"].value_counts())

Number of orders: 112932
Unique customers with orders: 13310

Order value:
count    112932.000000
mean         34.611150
std          20.049515
min           4.090000
25%          21.540000
50%          29.820000
75%          41.752500
max         322.830000
Name: order_value, dtype: float64

Discount rate:
0.2495484008075656

Orders by channel:
channel
App    84844
Web    28088
Name: count, dtype: int64

Orders by category:
category
Category B    22683
Category E    22667
Category C    22645
Category A    22616
Category D    22321
Name: count, dtype: int64


In [6]:
# Cell 6 — Create experiment assignments

experiment_date = pd.Timestamp("2026-09-30")
measurement_end = experiment_date + pd.Timedelta(days=30)

# One experiment record per customer
experiments = customers[["customer_id"]].copy()

experiments["experiment_id"] = np.arange(
    1,
    len(experiments) + 1
)

experiments["experiment_date"] = experiment_date

# Randomly assign customers to control or treatment
experiments["treatment_group"] = np.random.choice(
    ["Control", "Treatment"],
    size=len(experiments),
    p=[0.50, 0.50]
)

# Treatment receives $5 off $30
experiments["intervention"] = np.where(
    experiments["treatment_group"] == "Treatment",
    "$5 off $30",
    "No intervention"
)

experiments["offer_amount"] = np.where(
    experiments["treatment_group"] == "Treatment",
    5.00,
    0.00
)

experiments["minimum_spend"] = np.where(
    experiments["treatment_group"] == "Treatment",
    30.00,
    0.00
)

experiments.head()

,customer_id,experiment_id,experiment_date,treatment_group,intervention,offer_amount,minimum_spend
0,1,1,2026-09-30,Treatment,$5 off $30,5.0,30.0
1,2,2,2026-09-30,Treatment,$5 off $30,5.0,30.0
2,3,3,2026-09-30,Treatment,$5 off $30,5.0,30.0
3,4,4,2026-09-30,Treatment,$5 off $30,5.0,30.0
4,5,5,2026-09-30,Control,No intervention,0.0,0.0


In [7]:
# Cell 7 — Create historical customer behavior features

history = orders[
    orders["order_date"] < experiment_date
].copy()

customer_features = (
    history
    .groupby("customer_id")
    .agg(
        last_order_date=("order_date", "max"),
        total_orders=("order_id", "count"),
        total_spend=("net_order_value", "sum"),
        average_order_value=("net_order_value", "mean")
    )
    .reset_index()
)

# Recency
customer_features["days_since_last_order"] = (
    experiment_date - customer_features["last_order_date"]
).dt.days

# Orders in recent periods
orders_30d = (
    history[
        history["order_date"] >= experiment_date - pd.Timedelta(days=30)
    ]
    .groupby("customer_id")
    .size()
    .rename("orders_30d")
)

orders_60d = (
    history[
        (history["order_date"] >= experiment_date - pd.Timedelta(days=60)) &
        (history["order_date"] < experiment_date - pd.Timedelta(days=30))
    ]
    .groupby("customer_id")
    .size()
    .rename("orders_previous_30d")
)

orders_90d = (
    history[
        history["order_date"] >= experiment_date - pd.Timedelta(days=90)
    ]
    .groupby("customer_id")
    .size()
    .rename("orders_90d")
)

customer_features = (
    customer_features
    .merge(orders_30d, on="customer_id", how="left")
    .merge(orders_60d, on="customer_id", how="left")
    .merge(orders_90d, on="customer_id", how="left")
)

# Fill customers with no recent orders
for col in ["orders_30d", "orders_previous_30d", "orders_90d"]:
    customer_features[col] = customer_features[col].fillna(0)

# Frequency trend
customer_features["frequency_trend"] = (
    customer_features["orders_30d"]
    - customer_features["orders_previous_30d"]
)

customer_features.head()

,customer_id,last_order_date,total_orders,total_spend,average_order_value,days_since_last_order,orders_30d,orders_previous_30d,orders_90d,frequency_trend
0,2,2026-09-23 06:41:18,8,336.55,42.068750,6,1.0,0.0,1.0,1.0
1,4,2026-09-13 20:42:47,3,113.05,37.683333,16,2.0,1.0,3.0,1.0
2,5,2026-09-14 00:03:08,5,94.66,18.932000,15,1.0,2.0,4.0,-1.0
3,6,2026-09-19 00:00:00,6,231.51,38.585000,11,1.0,1.0,2.0,0.0
4,7,2026-09-25 00:00:00,23,757.15,32.919565,5,3.0,4.0,10.0,-1.0


In [8]:
# Cell 8 — Lifecycle segmentation

customer_features["segment"] = np.select(
    [
        customer_features["total_orders"] <= 1,

        (
            (customer_features["days_since_last_order"] <= 30) &
            (customer_features["orders_30d"] >= 2)
        ),

        (
            (customer_features["days_since_last_order"] > 30) &
            (customer_features["days_since_last_order"] <= 90) &
            (customer_features["total_orders"] >= 3)
        )
    ],
    [
        "New",
        "Active",
        "At Risk"
    ],
    default="Lapsed"
)

print(customer_features["segment"].value_counts())

segment
Active     5882
Lapsed     4109
At Risk    1832
New        1487
Name: count, dtype: int64


In [9]:
# Cell 9 — Create behavioral response probabilities

# Start with each customer's baseline purchase probability
customer_features["baseline_probability"] = (
    0.04
    + 0.015 * customer_features["orders_30d"]
    + 0.003 * customer_features["orders_90d"]
    - 0.002 * customer_features["days_since_last_order"]
)

# Keep probabilities in a reasonable range
customer_features["baseline_probability"] = (
    customer_features["baseline_probability"]
    .clip(0.02, 0.85)
)

# Treatment effect:
# At-risk customers get the strongest potential incremental response.
customer_features["treatment_lift"] = np.select(
    [
        customer_features["segment"] == "At Risk",
        customer_features["segment"] == "Active",
        customer_features["segment"] == "New",
        customer_features["segment"] == "Lapsed"
    ],
    [
        0.18,
        0.05,
        0.08,
        0.10
    ],
    default=0.05
)

# Add individual variation
customer_features["treatment_lift"] += np.random.normal(
    0,
    0.03,
    len(customer_features)
)

customer_features["treatment_lift"] = (
    customer_features["treatment_lift"]
    .clip(0, 0.30)
)

customer_features[
    [
        "customer_id",
        "segment",
        "baseline_probability",
        "treatment_lift"
    ]
].head(10)

,customer_id,segment,baseline_probability,treatment_lift
0,2,Lapsed,0.046,0.091788
1,4,Active,0.047,0.079507
2,5,Lapsed,0.037,0.124031
3,6,Lapsed,0.039,0.053209
4,7,Active,0.105,0.026407
5,8,Lapsed,0.052,0.100438
6,9,At Risk,0.020,0.181055
7,10,New,0.020,0.130626
8,11,New,0.020,0.085002
9,12,Active,0.039,0.045926


In [10]:
# Cell 10 — Generate post-intervention outcomes

experiments = experiments.merge(
    customer_features[
        [
            "customer_id",
            "segment",
            "baseline_probability",
            "treatment_lift"
        ]
    ],
    on="customer_id",
    how="left"
)

# Determine purchase probability based on treatment assignment
experiments["purchase_probability"] = np.where(
    experiments["treatment_group"] == "Treatment",
    experiments["baseline_probability"] + experiments["treatment_lift"],
    experiments["baseline_probability"]
)

# Keep probability between 0 and 1
experiments["purchase_probability"] = (
    experiments["purchase_probability"].clip(0, 0.95)
)

# Simulate conversion
experiments["converted"] = (
    np.random.random(len(experiments))
    < experiments["purchase_probability"]
).astype(int)

# Number of orders after intervention
experiments["post_intervention_orders"] = np.where(
    experiments["converted"] == 1,
    np.random.choice([1, 1, 1, 2, 2, 3], len(experiments)),
    0
)

# Post-intervention value
experiments["post_intervention_value"] = np.where(
    experiments["converted"] == 1,
    np.round(
        experiments["post_intervention_orders"]
        * np.random.lognormal(np.log(30), 0.35),
        2
    ),
    0
)

experiments.head()

,customer_id,experiment_id,experiment_date,treatment_group,intervention,offer_amount,minimum_spend,segment,baseline_probability,treatment_lift,purchase_probability,converted,post_intervention_orders,post_intervention_value
0,1,1,2026-09-30,Treatment,$5 off $30,5.0,30.0,NaN,NaN,NaN,NaN,0,0,0.0
1,2,2,2026-09-30,Treatment,$5 off $30,5.0,30.0,Lapsed,0.046,0.091788,0.137788,0,0,0.0
2,3,3,2026-09-30,Treatment,$5 off $30,5.0,30.0,NaN,NaN,NaN,NaN,0,0,0.0
3,4,4,2026-09-30,Treatment,$5 off $30,5.0,30.0,Active,0.047,0.079507,0.126507,0,0,0.0
4,5,5,2026-09-30,Control,No intervention,0.0,0.0,Lapsed,0.037,0.124031,0.037000,0,0,0.0


In [11]:
# Cell 11 — Experiment results

experiment_summary = (
    experiments
    .groupby("treatment_group")
    .agg(
        customers=("customer_id", "count"),
        conversion_rate=("converted", "mean"),
        total_orders=("post_intervention_orders", "sum"),
        total_value=("post_intervention_value", "sum")
    )
    .reset_index()
)

experiment_summary

,treatment_group,customers,conversion_rate,total_orders,total_value
0,Control,7409,0.055338,701,31027.02
1,Treatment,7591,0.125543,1618,71614.45


In [12]:
# Conversion rate by segment and treatment

segment_experiment = (
    experiments
    .groupby(["segment", "treatment_group"])
    .agg(
        customers=("customer_id", "count"),
        conversion_rate=("converted", "mean"),
        average_value=("post_intervention_value", "mean")
    )
    .reset_index()
)

segment_experiment

,segment,treatment_group,customers,conversion_rate,average_value
0,Active,Control,2905,0.103614,7.877112
1,Active,Treatment,2977,0.140410,10.570934
2,At Risk,Control,936,0.020299,1.702340
3,At Risk,Treatment,896,0.198661,15.017143
4,Lapsed,Control,2010,0.032338,2.290114
5,Lapsed,Treatment,2099,0.131015,9.699905
6,New,Control,727,0.034388,2.678803
7,New,Treatment,760,0.107895,8.328053


In [13]:
# Cell 12 — Build complete customer feature table

customer_features = (
    customers[
        [
            "customer_id",
            "signup_date",
            "acquisition_channel",
            "geography",
            "customer_tenure_days"
        ]
    ]
    .merge(
        customer_features,
        on="customer_id",
        how="left",
        suffixes=("", "_old")
    )
)

# Customers with no historical orders
customer_features["total_orders"] = (
    customer_features["total_orders"].fillna(0)
)

customer_features["total_spend"] = (
    customer_features["total_spend"].fillna(0)
)

customer_features["average_order_value"] = (
    customer_features["average_order_value"].fillna(0)
)

customer_features["orders_30d"] = (
    customer_features["orders_30d"].fillna(0)
)

customer_features["orders_previous_30d"] = (
    customer_features["orders_previous_30d"].fillna(0)
)

customer_features["orders_90d"] = (
    customer_features["orders_90d"].fillna(0)
)

customer_features["frequency_trend"] = (
    customer_features["frequency_trend"].fillna(0)
)

# No previous order = days since last order is based on experiment date
customer_features["days_since_last_order"] = (
    customer_features["days_since_last_order"]
    .fillna(
        (experiment_date - customer_features["signup_date"]).dt.days
    )
)

# Rebuild segment classification for ALL customers
customer_features["segment"] = np.select(
    [
        customer_features["total_orders"] == 0,

        customer_features["total_orders"] <= 1,

        (
            (customer_features["days_since_last_order"] <= 30) &
            (customer_features["orders_30d"] >= 2)
        ),

        (
            (customer_features["days_since_last_order"] > 30) &
            (customer_features["days_since_last_order"] <= 90) &
            (customer_features["total_orders"] >= 3)
        )
    ],
    [
        "New",
        "New",
        "Active",
        "At Risk"
    ],
    default="Lapsed"
)

print("Total customers:", len(customer_features))
print()
print(customer_features["segment"].value_counts())

Total customers: 15000

segment
Active     5882
Lapsed     4109
New        3177
At Risk    1832
Name: count, dtype: int64


In [14]:
# Cell 13 — Generate response probabilities for all customers

customer_features["baseline_probability"] = (
    0.04
    + 0.015 * customer_features["orders_30d"]
    + 0.003 * customer_features["orders_90d"]
    - 0.002 * customer_features["days_since_last_order"]
)

customer_features["baseline_probability"] = (
    customer_features["baseline_probability"]
    .clip(0.02, 0.85)
)

customer_features["treatment_lift"] = np.select(
    [
        customer_features["segment"] == "At Risk",
        customer_features["segment"] == "Active",
        customer_features["segment"] == "New",
        customer_features["segment"] == "Lapsed"
    ],
    [
        0.18,
        0.05,
        0.08,
        0.10
    ],
    default=0.05
)

customer_features["treatment_lift"] += np.random.normal(
    0,
    0.03,
    len(customer_features)
)

customer_features["treatment_lift"] = (
    customer_features["treatment_lift"]
    .clip(0, 0.30)
)

customer_features[
    [
        "customer_id",
        "segment",
        "orders_30d",
        "orders_90d",
        "days_since_last_order",
        "baseline_probability",
        "treatment_lift"
    ]
].head(10)

,customer_id,segment,orders_30d,orders_90d,days_since_last_order,baseline_probability,treatment_lift
0,1,New,0.0,0.0,96.0,0.020,0.049919
1,2,Lapsed,1.0,1.0,6.0,0.046,0.086334
2,3,New,0.0,0.0,95.0,0.020,0.053517
3,4,Active,2.0,3.0,16.0,0.047,0.040491
4,5,Lapsed,1.0,4.0,15.0,0.037,0.059663
5,6,Lapsed,1.0,2.0,11.0,0.039,0.090236
6,7,Active,3.0,10.0,5.0,0.105,0.055544
7,8,Lapsed,1.0,3.0,6.0,0.052,0.055945
8,9,At Risk,0.0,2.0,40.0,0.020,0.176705
9,10,New,0.0,1.0,51.0,0.020,0.087859


In [15]:
# Cell 14 — Attach complete customer features to experiment

# Remove the old feature columns from experiments
experiments = experiments.drop(
    columns=[
        "segment",
        "baseline_probability",
        "treatment_lift",
        "purchase_probability"
    ],
    errors="ignore"
)

# Attach the complete 15K customer feature set
experiments = experiments.merge(
    customer_features[
        [
            "customer_id",
            "segment",
            "baseline_probability",
            "treatment_lift"
        ]
    ],
    on="customer_id",
    how="left"
)

print("Experiment customers:", len(experiments))
print()
print(experiments[
    [
        "customer_id",
        "treatment_group",
        "segment",
        "baseline_probability",
        "treatment_lift"
    ]
].head())

Experiment customers: 15000

   customer_id treatment_group segment  baseline_probability  treatment_lift
0            1       Treatment     New                 0.020        0.049919
1            2       Treatment  Lapsed                 0.046        0.086334
2            3       Treatment     New                 0.020        0.053517
3            4       Treatment  Active                 0.047        0.040491
4            5         Control  Lapsed                 0.037        0.059663


In [16]:
# Cell 15 — Calculate incremental impact

control = experiments[
    experiments["treatment_group"] == "Control"
]

treatment = experiments[
    experiments["treatment_group"] == "Treatment"
]

# Conversion rates
control_rate = control["converted"].mean()
treatment_rate = treatment["converted"].mean()

# Expected conversions in treatment group
expected_treatment_conversions = (
    len(treatment) * control_rate
)

actual_treatment_conversions = (
    treatment["converted"].sum()
)

incremental_conversions = (
    actual_treatment_conversions
    - expected_treatment_conversions
)

# Conversion lift
incremental_conversion_rate = (
    treatment_rate - control_rate
)

# Expected treatment-group value without intervention
control_value_per_customer = (
    control["post_intervention_value"].sum()
    / len(control)
)

expected_treatment_value = (
    len(treatment) * control_value_per_customer
)

actual_treatment_value = (
    treatment["post_intervention_value"].sum()
)

incremental_value = (
    actual_treatment_value
    - expected_treatment_value
)

# Intervention cost
treatment_conversions = treatment["converted"].sum()

intervention_cost = (
    treatment_conversions * 5
)

# ROI
incremental_profit = (
    incremental_value - intervention_cost
)

roi = (
    incremental_profit / intervention_cost
)

print("CONTROL CONVERSION RATE:", round(control_rate * 100, 2), "%")
print("TREATMENT CONVERSION RATE:", round(treatment_rate * 100, 2), "%")
print()
print("INCREMENTAL CONVERSION RATE:", round(
    incremental_conversion_rate * 100, 2
), "percentage points")
print("INCREMENTAL CONVERSIONS:", round(incremental_conversions))
print()
print("EXPECTED TREATMENT VALUE:", round(expected_treatment_value, 2))
print("ACTUAL TREATMENT VALUE:", round(actual_treatment_value, 2))
print("INCREMENTAL VALUE:", round(incremental_value, 2))
print()
print("INTERVENTION COST:", round(intervention_cost, 2))
print("INCREMENTAL PROFIT:", round(incremental_profit, 2))
print("ROI:", round(roi * 100, 2), "%")

CONTROL CONVERSION RATE: 5.53 %
TREATMENT CONVERSION RATE: 12.55 %

INCREMENTAL CONVERSION RATE: 7.02 percentage points
INCREMENTAL CONVERSIONS: 533

EXPECTED TREATMENT VALUE: 31789.19
ACTUAL TREATMENT VALUE: 71614.45
INCREMENTAL VALUE: 39825.26

INTERVENTION COST: 4765
INCREMENTAL PROFIT: 35060.26
ROI: 735.79 %


In [17]:
# Cell 16 — Segment-level incremental impact

segment_results = []

for segment in experiments["segment"].dropna().unique():

    control_seg = experiments[
        (experiments["segment"] == segment) &
        (experiments["treatment_group"] == "Control")
    ]

    treatment_seg = experiments[
        (experiments["segment"] == segment) &
        (experiments["treatment_group"] == "Treatment")
    ]

    control_rate = control_seg["converted"].mean()
    treatment_rate = treatment_seg["converted"].mean()

    incremental_rate = (
        treatment_rate - control_rate
    )

    incremental_conversions = (
        len(treatment_seg) * incremental_rate
    )

    control_value_per_customer = (
        control_seg["post_intervention_value"].sum()
        / len(control_seg)
    )

    expected_treatment_value = (
        len(treatment_seg) * control_value_per_customer
    )

    actual_treatment_value = (
        treatment_seg["post_intervention_value"].sum()
    )

    incremental_value = (
        actual_treatment_value
        - expected_treatment_value
    )

    intervention_cost = (
        treatment_seg["converted"].sum() * 5
    )

    incremental_profit = (
        incremental_value - intervention_cost
    )

    segment_results.append({
        "segment": segment,
        "customers": len(treatment_seg),
        "control_rate": control_rate,
        "treatment_rate": treatment_rate,
        "incremental_lift": incremental_rate,
        "incremental_conversions": incremental_conversions,
        "incremental_value": incremental_value,
        "intervention_cost": intervention_cost,
        "incremental_profit": incremental_profit
    })

segment_results = pd.DataFrame(segment_results)

segment_results = segment_results.sort_values(
    "incremental_profit",
    ascending=False
).reset_index(drop=True)

segment_results

,segment,customers,control_rate,treatment_rate,incremental_lift,incremental_conversions,incremental_value,intervention_cost,incremental_profit
0,Lapsed,2099,0.032338,0.131015,0.098676,207.121891,15553.149816,1375,14178.149816
1,At Risk,896,0.020299,0.198661,0.178362,159.811966,11930.063590,890,11040.063590
2,Active,2977,0.103614,0.140410,0.036795,109.539759,8019.507945,2090,5929.507945
3,New,1619,0.016046,0.050649,0.034602,56.021181,4305.580392,410,3895.580392


In [18]:
# Cell 17 — Prepare ML modeling dataset

model_data = experiments.merge(
    customer_features[
        [
            "customer_id",
            "customer_tenure_days",
            "acquisition_channel",
            "geography",
            "orders_30d",
            "orders_previous_30d",
            "orders_90d",
            "total_orders",
            "total_spend",
            "average_order_value",
            "days_since_last_order",
            "frequency_trend",
            "segment"
        ]
    ],
    on="customer_id",
    how="left",
    suffixes=("", "_features")
)

model_data = model_data[
    [
        "customer_id",
        "treatment_group",
        "converted",
        "customer_tenure_days",
        "acquisition_channel",
        "geography",
        "orders_30d",
        "orders_previous_30d",
        "orders_90d",
        "total_orders",
        "total_spend",
        "average_order_value",
        "days_since_last_order",
        "frequency_trend",
        "segment"
    ]
]

print("Rows:", len(model_data))
print("Columns:", len(model_data.columns))

model_data.head()

Rows: 15000
Columns: 15


,customer_id,treatment_group,converted,customer_tenure_days,acquisition_channel,geography,orders_30d,orders_previous_30d,orders_90d,total_orders,total_spend,average_order_value,days_since_last_order,frequency_trend,segment
0,1,Treatment,0,96,Organic,South,0.0,0.0,0.0,0.0,0.00,0.000000,96.0,0.0,New
1,2,Treatment,0,178,Social,South,1.0,0.0,1.0,8.0,336.55,42.068750,6.0,1.0,Lapsed
2,3,Treatment,0,95,Social,Northeast,0.0,0.0,0.0,0.0,0.00,0.000000,95.0,0.0,New
3,4,Treatment,0,54,Organic,South,2.0,1.0,3.0,3.0,113.05,37.683333,16.0,1.0,Active
4,5,Control,0,120,Partnership,South,1.0,2.0,4.0,5.0,94.66,18.932000,15.0,-1.0,Lapsed


In [19]:
# Cell 18 — ML dataset sanity check

print("Rows:", len(model_data))
print("Missing values:")
print(model_data.isna().sum())

print("\nTreatment distribution:")
print(model_data["treatment_group"].value_counts())

print("\nConversion distribution:")
print(model_data["converted"].value_counts())

print("\nSegment distribution:")
print(model_data["segment"].value_counts())

Rows: 15000
Missing values:
customer_id              0
treatment_group          0
converted                0
customer_tenure_days     0
acquisition_channel      0
geography                0
orders_30d               0
orders_previous_30d      0
orders_90d               0
total_orders             0
total_spend              0
average_order_value      0
days_since_last_order    0
frequency_trend          0
segment                  0
dtype: int64

Treatment distribution:
treatment_group
Treatment    7591
Control      7409
Name: count, dtype: int64

Conversion distribution:
converted
0    13637
1     1363
Name: count, dtype: int64

Segment distribution:
segment
Active     5882
Lapsed     4109
New        3177
At Risk    1832
Name: count, dtype: int64


In [20]:
# Cell 19 — Prepare data for incremental response modeling

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# Features
feature_columns = [
    "customer_tenure_days",
    "acquisition_channel",
    "geography",
    "orders_30d",
    "orders_previous_30d",
    "orders_90d",
    "total_orders",
    "total_spend",
    "average_order_value",
    "days_since_last_order",
    "frequency_trend",
    "segment"
]

X = model_data[feature_columns]
y = model_data["converted"]

# Identify categorical and numerical columns
categorical_features = [
    "acquisition_channel",
    "geography",
    "segment"
]

numerical_features = [
    "customer_tenure_days",
    "orders_30d",
    "orders_previous_30d",
    "orders_90d",
    "total_orders",
    "total_spend",
    "average_order_value",
    "days_since_last_order",
    "frequency_trend"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

print("Features prepared.")


Features prepared.


In [21]:
# Cell 20 — Create treatment and control datasets

control_data = model_data[
    model_data["treatment_group"] == "Control"
].copy()

treatment_data = model_data[
    model_data["treatment_group"] == "Treatment"
].copy()

X_control = control_data[feature_columns]
y_control = control_data["converted"]

X_treatment = treatment_data[feature_columns]
y_treatment = treatment_data["converted"]

print("Control customers:", len(control_data))
print("Treatment customers:", len(treatment_data))
print()
print("Control conversions:", y_control.sum())
print("Treatment conversions:", y_treatment.sum())

Control customers: 7409
Treatment customers: 7591

Control conversions: 410
Treatment conversions: 953


In [22]:
# Cell 21 — Train separate response models

from sklearn.ensemble import RandomForestClassifier

# -------------------------
# CONTROL MODEL
# -------------------------

X_control_encoded = preprocessor.fit_transform(X_control)

control_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=20,
    random_state=42,
    class_weight="balanced"
)

control_model.fit(
    X_control_encoded,
    y_control
)


# -------------------------
# TREATMENT MODEL
# -------------------------

X_treatment_encoded = preprocessor.transform(X_treatment)

treatment_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=20,
    random_state=42,
    class_weight="balanced"
)

treatment_model.fit(
    X_treatment_encoded,
    y_treatment
)

print("Control model trained.")
print("Treatment model trained.")

Control model trained.
Treatment model trained.


In [23]:
# Cell 22 — Predict incremental response for every customer

X_all = model_data[feature_columns]

X_all_encoded = preprocessor.transform(X_all)

model_data["predicted_control_probability"] = (
    control_model.predict_proba(X_all_encoded)[:, 1]
)

model_data["predicted_treatment_probability"] = (
    treatment_model.predict_proba(X_all_encoded)[:, 1]
)

model_data["predicted_incremental_response"] = (
    model_data["predicted_treatment_probability"]
    - model_data["predicted_control_probability"]
)

model_data[
    [
        "customer_id",
        "segment",
        "predicted_control_probability",
        "predicted_treatment_probability",
        "predicted_incremental_response"
    ]
].head(10)

,customer_id,segment,predicted_control_probability,predicted_treatment_probability,predicted_incremental_response
0,1,New,0.000328,0.000328,-1.553286e-07
1,2,Lapsed,0.259123,0.504113,2.449909e-01
2,3,New,0.000000,0.004053,4.053371e-03
3,4,Active,0.318733,0.445230,1.264972e-01
4,5,Lapsed,0.305366,0.427406,1.220393e-01
5,6,Lapsed,0.279010,0.509667,2.306567e-01
6,7,Active,0.584369,0.496948,-8.742098e-02
7,8,Lapsed,0.318369,0.473310,1.549402e-01
8,9,At Risk,0.204419,0.541921,3.375015e-01
9,10,New,0.187553,0.434446,2.468931e-01


In [24]:
# Cell 23 — Train/test split

from sklearn.model_selection import train_test_split

control_train, control_test = train_test_split(
    control_data,
    test_size=0.25,
    random_state=42,
    stratify=control_data["converted"]
)

treatment_train, treatment_test = train_test_split(
    treatment_data,
    test_size=0.25,
    random_state=42,
    stratify=treatment_data["converted"]
)

print("Control train:", len(control_train))
print("Control test:", len(control_test))

print()

print("Treatment train:", len(treatment_train))
print("Treatment test:", len(treatment_test))

Control train: 5556
Control test: 1853

Treatment train: 5693
Treatment test: 1898


In [25]:
# Cell 24 — Train models using training data only

# Prepare training features
X_control_train = control_train[feature_columns]
y_control_train = control_train["converted"]

X_treatment_train = treatment_train[feature_columns]
y_treatment_train = treatment_train["converted"]

# Fit preprocessing ONLY on training data
preprocessor_ml = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

X_control_train_encoded = preprocessor_ml.fit_transform(
    X_control_train
)

X_treatment_train_encoded = preprocessor_ml.transform(
    X_treatment_train
)

# Control model
control_model_ml = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=20,
    random_state=42,
    class_weight="balanced"
)

control_model_ml.fit(
    X_control_train_encoded,
    y_control_train
)

# Treatment model
treatment_model_ml = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=20,
    random_state=42,
    class_weight="balanced"
)

treatment_model_ml.fit(
    X_treatment_train_encoded,
    y_treatment_train
)

print("Control model trained on training data.")
print("Treatment model trained on training data.")

Control model trained on training data.
Treatment model trained on training data.


In [26]:
# Cell 25 — Evaluate predictions on unseen customers

X_control_test = control_test[feature_columns]
X_treatment_test = treatment_test[feature_columns]

X_control_test_encoded = preprocessor_ml.transform(
    X_control_test
)

X_treatment_test_encoded = preprocessor_ml.transform(
    X_treatment_test
)

control_test["predicted_probability"] = (
    control_model_ml.predict_proba(
        X_control_test_encoded
    )[:, 1]
)

treatment_test["predicted_probability"] = (
    treatment_model_ml.predict_proba(
        X_treatment_test_encoded
    )[:, 1]
)

print("Control test predictions created.")
print("Treatment test predictions created.")

Control test predictions created.
Treatment test predictions created.


In [27]:
# Cell 26 — Evaluate model performance on unseen test data

from sklearn.metrics import roc_auc_score

control_auc = roc_auc_score(
    control_test["converted"],
    control_test["predicted_probability"]
)

treatment_auc = roc_auc_score(
    treatment_test["converted"],
    treatment_test["predicted_probability"]
)

print("Control Model ROC-AUC:", round(control_auc, 3))
print("Treatment Model ROC-AUC:", round(treatment_auc, 3))

Control Model ROC-AUC: 0.709
Treatment Model ROC-AUC: 0.603


In [28]:
# Cell 27 — Calculate out-of-sample incremental response

# Combine test customers
test_customers = pd.concat(
    [
        control_test,
        treatment_test
    ],
    ignore_index=True
)

# Use the same feature data for both models
X_test_all = test_customers[feature_columns]

X_test_all_encoded = preprocessor_ml.transform(
    X_test_all
)

# Predict what each customer would do under each scenario
test_customers["predicted_control_probability"] = (
    control_model_ml.predict_proba(
        X_test_all_encoded
    )[:, 1]
)

test_customers["predicted_treatment_probability"] = (
    treatment_model_ml.predict_proba(
        X_test_all_encoded
    )[:, 1]
)

# Incremental response
test_customers["predicted_incremental_response"] = (
    test_customers["predicted_treatment_probability"]
    - test_customers["predicted_control_probability"]
)

# Rank customers
test_customers = test_customers.sort_values(
    "predicted_incremental_response",
    ascending=False
).reset_index(drop=True)

test_customers[
    [
        "customer_id",
        "segment",
        "predicted_control_probability",
        "predicted_treatment_probability",
        "predicted_incremental_response"
    ]
].head(20)

,customer_id,segment,predicted_control_probability,predicted_treatment_probability,predicted_incremental_response
0,4988,At Risk,0.085862,0.609812,0.523950
1,146,At Risk,0.092696,0.602583,0.509887
2,8740,At Risk,0.096911,0.596174,0.499263
3,3605,At Risk,0.126467,0.621790,0.495323
4,12017,At Risk,0.157344,0.645857,0.488514
5,6213,At Risk,0.137354,0.625601,0.488247
6,10620,At Risk,0.127276,0.613499,0.486223
7,1622,At Risk,0.118053,0.601645,0.483592
8,14496,At Risk,0.116126,0.596929,0.480803
9,7849,At Risk,0.148656,0.625412,0.476756


In [29]:
# Cell 28 — Create incremental-response deciles

test_customers["uplift_decile"] = pd.qcut(
    test_customers["predicted_incremental_response"],
    q=10,
    labels=False,
    duplicates="drop"
)

# Make highest predicted response = Decile 10
test_customers["uplift_decile"] = (
    10 - test_customers["uplift_decile"]
)

test_customers[
    [
        "uplift_decile",
        "predicted_incremental_response"
    ]
].groupby("uplift_decile").agg(
    customers=("predicted_incremental_response", "count"),
    avg_predicted_uplift=("predicted_incremental_response", "mean")
).sort_index()

,customers,avg_predicted_uplift
uplift_decile,,
1,375,0.395325
2,375,0.316305
3,375,0.274518
4,375,0.231694
5,375,0.162319
6,375,0.059461
7,375,0.002009
8,375,-0.021390
9,375,-0.085237


In [30]:
# Cell 29 — Validate predicted uplift against actual experiment results

uplift_validation = (
    test_customers
    .groupby(["uplift_decile", "treatment_group"])
    .agg(
        customers=("customer_id", "count"),
        conversions=("converted", "sum"),
        conversion_rate=("converted", "mean")
    )
    .reset_index()
)

uplift_validation

,uplift_decile,treatment_group,customers,conversions,conversion_rate
0,1,Control,194,2,0.010309
1,1,Treatment,181,25,0.138122
2,2,Control,188,7,0.037234
3,2,Treatment,187,24,0.128342
4,3,Control,178,7,0.039326
5,3,Treatment,197,31,0.157360
6,4,Control,189,10,0.052910
7,4,Treatment,186,23,0.123656
8,5,Control,186,5,0.026882
9,5,Treatment,189,25,0.132275


In [31]:
# Cell 30 — Clean predicted vs actual uplift validation

validation_summary = (
    test_customers
    .groupby(["uplift_decile", "treatment_group"])
    .agg(
        customers=("customer_id", "count"),
        conversions=("converted", "sum"),
        conversion_rate=("converted", "mean")
    )
    .reset_index()
)

validation_pivot = validation_summary.pivot(
    index="uplift_decile",
    columns="treatment_group",
    values=["customers", "conversion_rate"]
)

validation_pivot.columns = [
    f"{metric}_{group.lower()}"
    for metric, group in validation_pivot.columns
]

validation_pivot = validation_pivot.reset_index()

validation_pivot["actual_incremental_lift"] = (
    validation_pivot["conversion_rate_treatment"]
    - validation_pivot["conversion_rate_control"]
)

predicted_uplift = (
    test_customers
    .groupby("uplift_decile")["predicted_incremental_response"]
    .mean()
    .reset_index(name="predicted_incremental_lift")
)

validation_summary_final = validation_pivot.merge(
    predicted_uplift,
    on="uplift_decile"
)

validation_summary_final

,uplift_decile,customers_control,customers_treatment,conversion_rate_control,conversion_rate_treatment,actual_incremental_lift,predicted_incremental_lift
0,1,194.0,181.0,0.010309,0.138122,0.127812,0.395325
1,2,188.0,187.0,0.037234,0.128342,0.091108,0.316305
2,3,178.0,197.0,0.039326,0.157360,0.118035,0.274518
3,4,189.0,186.0,0.052910,0.123656,0.070746,0.231694
4,5,186.0,189.0,0.026882,0.132275,0.105393,0.162319
5,6,186.0,189.0,0.064516,0.121693,0.057177,0.059461
6,7,162.0,213.0,0.006173,0.032864,0.026691,0.002009
7,8,190.0,185.0,0.052632,0.102703,0.050071,-0.021390
8,9,197.0,178.0,0.106599,0.162921,0.056322,-0.085237
9,10,183.0,193.0,0.153005,0.165803,0.012798,-0.165088


In [34]:
# Cell 31 — Rebuild uplift groups from predicted response

test_customers["uplift_group"] = pd.qcut(
    test_customers["predicted_incremental_response"],
    q=10,
    labels=False
)

# Highest predicted uplift = Group 1
test_customers["uplift_group"] = (
    10 - test_customers["uplift_group"]
)

test_customers[
    [
        "customer_id",
        "uplift_group",
        "predicted_incremental_response"
    ]
].head(10)

,customer_id,uplift_group,predicted_incremental_response
0,4988,1,0.523950
1,146,1,0.509887
2,8740,1,0.499263
3,3605,1,0.495323
4,12017,1,0.488514
5,6213,1,0.488247
6,10620,1,0.486223
7,1622,1,0.483592
8,14496,1,0.480803
9,7849,1,0.476756


In [35]:
test_customers.groupby("uplift_group").agg(
    customers=("customer_id", "count"),
    avg_predicted_uplift=("predicted_incremental_response", "mean"),
    min_predicted_uplift=("predicted_incremental_response", "min"),
    max_predicted_uplift=("predicted_incremental_response", "max")
).sort_index()

,customers,avg_predicted_uplift,min_predicted_uplift,max_predicted_uplift
uplift_group,,,,
1,375,0.395325,0.346777,0.523950
2,375,0.316305,0.294163,0.346601
3,375,0.274518,0.255181,0.294071
4,375,0.231694,0.203615,0.255088
5,375,0.162319,0.106682,0.203205
6,375,0.059461,0.017559,0.106437
7,375,0.002009,-0.003475,0.017440
8,375,-0.021390,-0.052059,-0.003551
9,375,-0.085237,-0.117488,-0.052298


In [38]:
# Cell 33A — Attach experiment outcome values

test_customers = test_customers.merge(
    experiments[
        [
            "customer_id",
            "post_intervention_value"
        ]
    ],
    on="customer_id",
    how="left"
)

print(
    "Missing post-intervention values:",
    test_customers["post_intervention_value"].isna().sum()
)

Missing post-intervention values: 0


In [39]:
# Cell 33B — Targeting policy economics

targeting_levels = {
    "Top 10%": 1,
    "Top 20%": 2,
    "Top 30%": 3,
    "Top 50%": 5,
    "Top 70%": 7,
    "Everyone": 10
}

results = []

for strategy, max_group in targeting_levels.items():

    targeted = test_customers[
        test_customers["uplift_group"] <= max_group
    ]

    control = targeted[
        targeted["treatment_group"] == "Control"
    ]

    treatment = targeted[
        targeted["treatment_group"] == "Treatment"
    ]

    control_conversion = control["converted"].mean()
    treatment_conversion = treatment["converted"].mean()

    incremental_lift = (
        treatment_conversion - control_conversion
    )

    incremental_conversions = (
        incremental_lift * len(treatment)
    )

    # Actual observed value
    control_value = control["post_intervention_value"].mean()
    treatment_value = treatment["post_intervention_value"].mean()

    incremental_value_per_customer = (
        treatment_value - control_value
    )

    incremental_value = (
        incremental_value_per_customer * len(treatment)
    )

    # $5 cost when the offer converts
    intervention_cost = (
        treatment["converted"].sum() * 5
    )

    incremental_profit = (
        incremental_value - intervention_cost
    )

    roi = (
        incremental_profit / intervention_cost
        if intervention_cost > 0
        else np.nan
    )

    results.append({
        "strategy": strategy,
        "customers_targeted": len(targeted),
        "control_conversion": control_conversion,
        "treatment_conversion": treatment_conversion,
        "actual_incremental_lift": incremental_lift,
        "incremental_conversions": incremental_conversions,
        "incremental_value": incremental_value,
        "intervention_cost": intervention_cost,
        "incremental_profit": incremental_profit,
        "ROI": roi
    })

targeting_economics = pd.DataFrame(results)

targeting_economics

,strategy,customers_targeted,control_conversion,treatment_conversion,actual_incremental_lift,incremental_conversions,incremental_value,intervention_cost,incremental_profit,ROI
0,Top 10%,375,0.010309,0.138122,0.127812,23.134021,1956.387629,125,1831.387629,14.651101
1,Top 20%,750,0.023560,0.133152,0.109592,40.329843,3475.099686,245,3230.099686,13.184080
2,Top 30%,1125,0.028571,0.141593,0.113021,63.857143,5302.260804,400,4902.260804,12.255652
3,Top 50%,1875,0.033155,0.136170,0.103015,96.834225,7113.465401,640,6473.465401,10.114790
4,Top 70%,2625,0.034295,0.117735,0.083440,111.976617,8293.013157,790,7503.013157,9.497485
5,Everyone,3751,0.055586,0.125395,0.069810,132.498651,9590.311074,1190,8400.311074,7.059085


In [42]:
# Create the final dataset for Power BI

powerbi_data = test_customers.copy()

# Create a simple readable targeting recommendation
powerbi_data["target_recommendation"] = np.where(
    powerbi_data["uplift_group"] <= 3,
    "Target",
    "Do Not Target"
)

# Create a simple uplift band
powerbi_data["uplift_band"] = pd.cut(
    powerbi_data["predicted_incremental_response"],
    bins=[-np.inf, 0, 0.10, 0.20, np.inf],
    labels=[
        "Negative",
        "Low",
        "Medium",
        "High"
    ]
)

# Select the columns we want in Power BI
powerbi_data = powerbi_data[
    [
        "customer_id",
        "segment",
        "acquisition_channel",
        "geography",
        "customer_tenure_days",
        "orders_30d",
        "orders_previous_30d",
        "orders_90d",
        "total_orders",
        "total_spend",
        "average_order_value",
        "days_since_last_order",
        "frequency_trend",
        "treatment_group",
        "converted",
        "post_intervention_value",
        "predicted_control_probability",
        "predicted_treatment_probability",
        "predicted_incremental_response",
        "uplift_group",
        "targeting_tier",
        "uplift_band",
        "target_recommendation"
    ]
]

print("Rows:", len(powerbi_data))
print("Columns:", len(powerbi_data.columns))

powerbi_data.head()

Rows: 3751
Columns: 23


,customer_id,segment,acquisition_channel,geography,customer_tenure_days,orders_30d,orders_previous_30d,orders_90d,total_orders,total_spend,...,treatment_group,converted,post_intervention_value,predicted_control_probability,predicted_treatment_probability,predicted_incremental_response,uplift_group,targeting_tier,uplift_band,target_recommendation
0,4988,At Risk,Partnership,Midwest,272,0.0,0.0,3.0,4.0,127.98,...,Control,0,0.0,0.085862,0.609812,0.523950,1,9,High,Target
1,146,At Risk,Partnership,West,320,0.0,1.0,2.0,5.0,144.44,...,Control,0,0.0,0.092696,0.602583,0.509887,1,9,High,Target
2,8740,At Risk,Referral,Northeast,280,0.0,0.0,1.0,3.0,94.66,...,Treatment,0,0.0,0.096911,0.596174,0.499263,1,9,High,Target
3,3605,At Risk,Referral,Midwest,309,0.0,0.0,1.0,4.0,98.26,...,Control,0,0.0,0.126467,0.621790,0.495323,1,9,High,Target
4,12017,At Risk,Referral,West,222,0.0,0.0,1.0,5.0,184.73,...,Treatment,0,0.0,0.157344,0.645857,0.488514,1,9,High,Target


In [43]:
# Save the final Power BI dataset as a CSV file

powerbi_data.to_csv(
    "consumer_growth_powerbi.csv",
    index=False
)

print("Power BI dataset saved successfully!")

Power BI dataset saved successfully!


In [44]:
# Build the full 15K Power BI base dataset

powerbi_full = customer_features.merge(
    experiments[
        [
            "customer_id",
            "treatment_group",
            "intervention",
            "offer_amount",
            "minimum_spend",
            "converted",
            "post_intervention_orders",
            "post_intervention_value"
        ]
    ],
    on="customer_id",
    how="left"
)

print("Rows:", len(powerbi_full))
print("Columns:", len(powerbi_full.columns))

print("\nMissing values:")
print(powerbi_full.isna().sum())

Rows: 15000
Columns: 24

Missing values:
customer_id                    0
signup_date                    0
acquisition_channel            0
geography                      0
customer_tenure_days           0
last_order_date             1690
total_orders                   0
total_spend                    0
average_order_value            0
days_since_last_order          0
orders_30d                     0
orders_previous_30d            0
orders_90d                     0
frequency_trend                0
segment                        0
baseline_probability           0
treatment_lift                 0
treatment_group                0
intervention                   0
offer_amount                   0
minimum_spend                  0
converted                      0
post_intervention_orders       0
post_intervention_value        0
dtype: int64


In [45]:
# Remove hidden synthetic variables
# These were used only to generate the synthetic experiment.

powerbi_full = powerbi_full.drop(
    columns=[
        "baseline_probability",
        "treatment_lift"
    ]
)

print("Rows:", len(powerbi_full))
print("Columns:", len(powerbi_full.columns))
print("\nRemaining columns:")
print(powerbi_full.columns.tolist())

Rows: 15000
Columns: 22

Remaining columns:
['customer_id', 'signup_date', 'acquisition_channel', 'geography', 'customer_tenure_days', 'last_order_date', 'total_orders', 'total_spend', 'average_order_value', 'days_since_last_order', 'orders_30d', 'orders_previous_30d', 'orders_90d', 'frequency_trend', 'segment', 'treatment_group', 'intervention', 'offer_amount', 'minimum_spend', 'converted', 'post_intervention_orders', 'post_intervention_value']


In [49]:
print("Control model features:")
print(control_model.feature_names_in_)

print("\nTreatment model features:")
print(treatment_model.feature_names_in_)

Control model features:


AttributeError: 'RandomForestClassifier' object has no attribute 'feature_names_in_'

In [50]:
print([name for name in globals() if "preprocess" in name.lower() or "transform" in name.lower()])

['ColumnTransformer', 'preprocessor', 'preprocessor_ml']


In [51]:
print(preprocessor_ml)

ColumnTransformer(transformers=[('categorical',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['acquisition_channel', 'geography',
                                  'segment']),
                                ('numerical', 'passthrough',
                                 ['customer_tenure_days', 'orders_30d',
                                  'orders_previous_30d', 'orders_90d',
                                  'total_orders', 'total_spend',
                                  'average_order_value',
                                  'days_since_last_order',
                                  'frequency_trend'])])


In [52]:
# Features used by our ML models

model_features = [
    "customer_tenure_days",
    "acquisition_channel",
    "geography",
    "orders_30d",
    "orders_previous_30d",
    "orders_90d",
    "total_orders",
    "total_spend",
    "average_order_value",
    "days_since_last_order",
    "frequency_trend",
    "segment"
]

# Get the model input data for all 15,000 customers
X_full = powerbi_full[model_features]

# Apply the same preprocessing used when training the models
X_full_transformed = preprocessor_ml.transform(X_full)

print("Original shape:", X_full.shape)
print("Transformed shape:", X_full_transformed.shape)

# Predict conversion probability WITHOUT the intervention
powerbi_full["predicted_control_probability"] = (
    control_model.predict_proba(X_full_transformed)[:, 1]
)

# Predict conversion probability WITH the intervention
powerbi_full["predicted_treatment_probability"] = (
    treatment_model.predict_proba(X_full_transformed)[:, 1]
)

# Predicted incremental response
powerbi_full["predicted_incremental_response"] = (
    powerbi_full["predicted_treatment_probability"]
    - powerbi_full["predicted_control_probability"]
)

print("\nPrediction summary:")
print(
    powerbi_full[
        [
            "predicted_control_probability",
            "predicted_treatment_probability",
            "predicted_incremental_response"
        ]
    ].describe()
)

Original shape: (15000, 12)
Transformed shape: (15000, 22)

Prediction summary:
       predicted_control_probability  predicted_treatment_probability  \
count                   15000.000000                     15000.000000   
mean                        0.338950                         0.440571   
std                         0.201768                         0.169477   
min                         0.000000                         0.000026   
25%                         0.210810                         0.442644   
50%                         0.300050                         0.487582   
75%                         0.489581                         0.524550   
max                         0.814678                         0.769474   

       predicted_incremental_response  
count                    15000.000000  
mean                         0.101621  
std                          0.166705  
min                         -0.323970  
25%                         -0.009369  
50%                   

In [53]:
# Rank customers into 10 uplift groups
# Group 1 = highest predicted incremental response

powerbi_full["uplift_group"] = pd.qcut(
    powerbi_full["predicted_incremental_response"],
    q=10,
    labels=False,
    duplicates="drop"
) + 1

# Create a simple targeting recommendation
powerbi_full["target_recommendation"] = np.where(
    powerbi_full["uplift_group"] <= 3,
    "Target",
    "Do Not Target"
)

# Create easy-to-read uplift bands
powerbi_full["uplift_band"] = pd.cut(
    powerbi_full["predicted_incremental_response"],
    bins=[-np.inf, 0, 0.10, 0.20, np.inf],
    labels=[
        "Negative",
        "Low",
        "Medium",
        "High"
    ]
)

print("Customers by uplift group:")
print(powerbi_full["uplift_group"].value_counts().sort_index())

print("\nTarget recommendation:")
print(powerbi_full["target_recommendation"].value_counts())

print("\nUplift bands:")
print(powerbi_full["uplift_band"].value_counts().sort_index())

Customers by uplift group:
uplift_group
1     1500
2     1500
3     1501
4     1499
5     1500
6     1500
7     1500
8     1500
9     1500
10    1500
Name: count, dtype: int64

Target recommendation:
target_recommendation
Do Not Target    10499
Target            4501
Name: count, dtype: int64

Uplift bands:
uplift_band
Negative    4641
Low         2989
Medium      2678
High        4692
Name: count, dtype: int64


In [54]:
# Create the final Power BI dataset

powerbi_final = powerbi_full[
    [
        # Customer information
        "customer_id",
        "segment",
        "acquisition_channel",
        "geography",
        "signup_date",
        "customer_tenure_days",

        # Historical behavior
        "last_order_date",
        "orders_30d",
        "orders_previous_30d",
        "orders_90d",
        "total_orders",
        "total_spend",
        "average_order_value",
        "days_since_last_order",
        "frequency_trend",

        # Experiment
        "treatment_group",
        "intervention",
        "offer_amount",
        "minimum_spend",
        "converted",
        "post_intervention_orders",
        "post_intervention_value",

        # ML targeting
        "predicted_control_probability",
        "predicted_treatment_probability",
        "predicted_incremental_response",
        "uplift_group",
        "uplift_band",
        "target_recommendation"
    ]
].copy()

print("Rows:", len(powerbi_final))
print("Columns:", len(powerbi_final.columns))

print("\nFinal columns:")
print(powerbi_final.columns.tolist())

Rows: 15000
Columns: 28

Final columns:
['customer_id', 'segment', 'acquisition_channel', 'geography', 'signup_date', 'customer_tenure_days', 'last_order_date', 'orders_30d', 'orders_previous_30d', 'orders_90d', 'total_orders', 'total_spend', 'average_order_value', 'days_since_last_order', 'frequency_trend', 'treatment_group', 'intervention', 'offer_amount', 'minimum_spend', 'converted', 'post_intervention_orders', 'post_intervention_value', 'predicted_control_probability', 'predicted_treatment_probability', 'predicted_incremental_response', 'uplift_group', 'uplift_band', 'target_recommendation']


In [55]:
# Export the final full customer dataset for Power BI

powerbi_final.to_csv(
    "consumer_growth_powerbi_full.csv",
    index=False
)

print("Power BI dataset saved successfully!")
print("Rows:", len(powerbi_final))


Power BI dataset saved successfully!
Rows: 15000


In [56]:

# Check which models and preprocessing objects exist
# in your current Jupyter notebook.

objects_to_check = [
    "control_model",
    "treatment_model",
    "preprocessor_ml",
    "features",
    "X_train_control",
    "X_train_treatment",
    "powerbi_full"
]

for name in objects_to_check:
    print(f"{name}: {'Available' if name in globals() else 'Not available'}")


control_model: Available
treatment_model: Available
preprocessor_ml: Available
features: Not available
X_train_control: Not available
X_train_treatment: Not available
powerbi_full: Available
